In [ ]:
# RAG Evaluation pipeline using Gemma 3 (4B Instruct) as LLM Judge in Google Colab
# 1. Install required libraries for Transformers, BitsAndBytes, and data processing
# Install libraries for Transformers, BitsAndBytes, and data management
!pip install -q transformers accelerate bitsandbytes pandas tqdm


In [ ]:
# 2. Authenticate with Hugging Face Hub using Colab Secrets (HF_TOKEN) or manual login
from huggingface_hub import login
from google.colab import userdata

# Authenticate to Hugging Face Hub via Colab Secret (HF_TOKEN) or manually
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Η σύνδεση στο Hugging Face πραγματοποιήθηκε επιτυχώς.")
except Exception:
    print("Δεν βρέθηκε HF_TOKEN στα Secrets. Εισάγετε το token χειροκίνητα:")
    login()


In [ ]:
# 3. Load Gemma 3 4B Instruct model with 4-bit quantization on GPU
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Define Gemma 3 4B Instruct model
MODEL_ID = "google/gemma-3-4b-it"

print(f"Φόρτωση μοντέλου: {MODEL_ID}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Δεν βρέθηκε GPU!'}")

# Configure 4-bit quantization to save GPU VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load tokenizer and model onto GPU
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Το μοντέλο φορτώθηκε επιτυχώς.")


In [ ]:
# 4. Load and prepare evaluation dataset and category mapping helpers
import json
import re
import pandas as pd
from google.colab import files

# Select JSON load method
LOAD_METHOD = "upload"  # "upload" ή "path"
JSON_PATH = "/content/rag_evaluation_data.json"

if LOAD_METHOD == "upload":
    print("Επιλέξτε το αρχείο JSON:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    data = json.loads(uploaded[filename])
else:
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

# Map JSON question types to internal category keys
QUESTION_TYPE_MAP = {
    "factual recall":        "factual",
    "conceptual understanding": "conceptual",
    "multi-hop reasoning":   "multi_hop",
    "unanswerable":          "unanswerable",
    "image-based":           "image_based",
}

def map_category(item: dict) -> str:
    """Maps the question_type JSON field to an internal category key."""
    raw_type = item.get("question_type") or "Factual Recall"
    key = raw_type.strip().lower()
    return QUESTION_TYPE_MAP.get(key, "factual")
def split_context_chunks(retrieved_context: str) -> list:
    """Splits retrieved context text into individual chunks based on source headers."""
    if not retrieved_context:
        return []
    raw_chunks = re.split(r'(?=Πηγή:)', retrieved_context.strip())
    return [c.strip() for c in raw_chunks if c.strip()]

# Process and prepare categories and chunks for each question
for item in data:
    item["_category"] = map_category(item)
    ctx = item.get("retrieved_context", "")
    item["_chunks"] = split_context_chunks(ctx)
    item["_has_image_chunks"] = any("[ΕΙΚΟΝΑ" in c for c in item["_chunks"])

print(f"Φορτώθηκαν {len(data)} ερωτήσεις.")
df_preview = pd.DataFrame([{
    "#": i+1,
    "query": item["query"][:70] + "...",
    "question_type": item.get("question_type", ""),
    "category": item["_category"],
    "chunks": len(item["_chunks"]),
    "έχει εικόνες": item["_has_image_chunks"]
} for i, item in enumerate(data)])
print(df_preview.to_string(index=False))


In [ ]:
# 5. Define evaluation prompts and criteria for Faithfulness, Relevance, and Context Quality
# Prompt for Faithfulness evaluation
FAITHFULNESS_PROMPT = """Είσαι αυστηρός αξιολογητής RAG συστημάτων. Αξιολόγησε την ΠΙΣΤΟΤΗΤΑ (Faithfulness) της απάντησης.

ΟΡΙΣΜΟΣ: Faithfulness = κάθε ισχυρισμός της απάντησης υποστηρίζεται ρητά από κάποιο chunk. Αν το σύστημα εφηύρε πληροφορίες που δεν υπάρχουν στα chunks, το σκορ πέφτει.

ΕΡΩΤΗΣΗ: {question}

RETRIEVED CHUNKS:
{chunks}

ΑΠΑΝΤΗΣΗ ΣΥΣΤΗΜΑΤΟΣ: {answer}

{special_instructions}

ΟΔΗΓΙΕΣ:
1. Γράψε σύντομη ανάλυση (2-4 προτάσεις): ποιοι ισχυρισμοί της απάντησης υποστηρίζονται από chunks και ποιοι όχι.
2. Μετά δώσε το τελικό σκορ.

ΣΚΟΡ (1-5):
5 = Όλοι οι ισχυρισμοί υποστηρίζονται πλήρως
4 = Σχεδόν όλοι υποστηρίζονται, μικρές αποκλίσεις
3 = Μερική υποστήριξη, μερικοί ισχυρισμοί δεν τεκμηριώνονται
2 = Σημαντικό hallucination
1 = Η απάντηση αντιφάσκει ή αγνοεί εντελώς τα chunks

Απάντησε ΜΟΝΟ με αυτή τη μορφή:
ΑΝΑΛΥΣΗ: [σύντομη ανάλυση]
ΣΚΟΡ: [αριθμός 1-5]"""

# Prompt for Answer Relevance evaluation
RELEVANCE_PROMPT = """Είσαι αυστηρός αξιολογητής RAG συστημάτων. Αξιολόγησε τη ΣΧΕΤΙΚΟΤΗΤΑ (Answer Relevance) της απάντησης.

ΟΡΙΣΜΟΣ: Answer Relevance = η απάντηση ικανοποιεί πραγματικά την ερώτηση. Ακόμα κι αν είναι factually σωστή, μπορεί να είναι εκτός θέματος.

ΕΡΩΤΗΣΗ: {question}

ΑΠΑΝΤΗΣΗ ΣΥΣΤΗΜΑΤΟΣ: {answer}

{special_instructions}

ΟΔΗΓΙΕΣ:
1. Γράψε σύντομη ανάλυση: τι ρωτά η ερώτηση, τι απαντά το σύστημα, αν ταυτίζονται.
2. Μετά δώσε το τελικό σκορ.

ΣΚΟΡ (1-5):
5 = Απαντά ακριβώς στην ερώτηση, πλήρης και στοχευμένη
4 = Απαντά σε μεγάλο βαθμό, μικρά κενά
3 = Μερική απάντηση, αγγίζει το θέμα αλλά χάνει κύρια σημεία
2 = Εκτός θέματος σε μεγάλο βαθμό
1 = Δεν απαντά καθόλου

Απάντησε ΜΟΝΟ με αυτή τη μορφή:
ΑΝΑΛΥΣΗ: [σύντομη ανάλυση]
ΣΚΟΡ: [αριθμός 1-5]"""

# Prompt for Retrieval Quality evaluation (Context Precision & Recall)
CONTEXT_PROMPT = """Είσαι αυστηρός αξιολογητής RAG συστημάτων. Αξιολόγησε την ΠΟΙΟΤΗΤΑ RETRIEVAL (Context Precision & Recall).

ΟΡΙΣΜΟΣ:
- Context Precision: από τα chunks που ανακτήθηκαν, πόσα ήταν χρήσιμα;
- Context Recall: υπήρχε η απαραίτητη πληροφορία στα chunks, ή έλειπε;

ΕΡΩΤΗΣΗ: {question}

RETRIEVED CHUNKS:
{chunks}

{special_instructions}

ΟΔΗΓΙΕΣ:
1. Γράψε σύντομη ανάλυση:
   - Ποια chunks ήταν χρήσιμα και ποια άσχετα (Precision)
   - Αν τα chunks περιείχαν αρκετή πληροφορία (Recall)
2. Μετά δώσε το τελικό σκορ.

ΣΚΟΡ (1-5):
5 = Όλα τα chunks σχετικά, πλήρης πληροφορία
4 = Κυρίως σχετικά, η πληροφορία υπάρχει
3 = Μικτά αποτελέσματα - μερικά σχετικά ή ελλιπής πληροφορία
2 = Κυρίως άσχετα chunks ή σημαντικές ελλείψεις
1 = Τα chunks δεν περιέχουν καμία σχετική πληροφορία (retrieval failure)

Απάντησε ΜΟΝΟ με αυτή τη μορφή:
ΑΝΑΛΥΣΗ: [σύντομη ανάλυση]
ΣΚΟΡ: [αριθμός 1-5]"""

# Special evaluation instructions per question category
SPECIAL_INSTRUCTIONS = {
    "unanswerable": """ΕΙΔΙΚΗ ΟΔΗΓΙΑ - UNANSWERABLE QUESTION:
Αυτή η ερώτηση είναι εκτός του περιεχομένου του βιβλίου. Το ΣΩΣΤΟ αποτέλεσμα είναι η άρνηση απάντησης ("δεν γνωρίζω", "δεν βρίσκεται στο βιβλίο", κλπ.).
Faithfulness: 5 αν αρνήθηκε, 1 αν εφηύρε απάντηση.
Answer Relevance: 5 αν αναγνώρισε ότι η ερώτηση δεν απαντάται.""",

    "image_based": """ΕΙΔΙΚΗ ΟΔΗΓΙΑ - IMAGE-BASED QUESTION:
Η πηγή πληροφορίας είναι η αυτόματη περιγραφή εικόνας από το moondream (vision model), ΟΧΙ το πρωτότυπο διάγραμμα. Κρίνε αν η απάντηση συνάδει με αυτό που περιέγραψε το moondream στα chunks, λαμβάνοντας υπόψη ότι οι περιγραφές μπορεί να είναι ατελείς.""",

    "multi_hop": """ΕΙΔΙΚΗ ΟΔΗΓΙΑ - MULTI-HOP QUESTION:
Αυτή η ερώτηση απαιτεί σύνθεση πληροφοριών από ΠΟΛΛΑ chunks. Ελέγξε αν η απάντηση συνδυάζει στοιχεία από διαφορετικά chunks, ή αν βασίστηκε μόνο σε ένα.""",

    "factual": "",
    "conceptual": "",
}

print("Τα prompts φορτώθηκαν επιτυχώς.")


In [ ]:
# 6. Helper execution, score parsing, and context truncation functions
def run_judge(prompt_text: str, max_new_tokens: int = 512) -> str:
    """Sends prompt to Gemma 3 model and returns response."""
    messages = [{"role": "user", "content": prompt_text}]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_length = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][prompt_length:]

    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def parse_score_and_analysis(raw_output: str) -> tuple:
    """Extracts analysis and score (1-5) from model output."""
    analysis = "(δεν αναλύθηκε)"
    score = -1

    analysis_match = re.search(r'ΑΝΑΛΥΣΗ:\s*(.+?)(?=ΣΚΟΡ:|$)', raw_output, re.DOTALL | re.IGNORECASE)
    if analysis_match:
        analysis = analysis_match.group(1).strip()

    score_match = re.search(r'ΣΚΟΡ:\s*([1-5])', raw_output, re.IGNORECASE)
    if score_match:
        score = int(score_match.group(1))
    else:
        numbers = re.findall(r'\b([1-5])\b', raw_output[-150:])
        if numbers:
            score = int(numbers[-1])

    return analysis, score

def prepare_context_for_prompt(retrieved_context: str) -> str:
    """Formats retrieved chunks with numbering for prompt insertion."""
    raw_chunks = split_context_chunks(retrieved_context)
    numbered = []
    for i, chunk in enumerate(raw_chunks, 1):
        first_line = chunk.split("\n")[0]
        chunk_type = "ΕΙΚΟΝΑ" if "[ΕΙΚΟΝΑ" in first_line else "ΚΕΙΜΕΝΟ"
        numbered.append(f"Chunk {i} ({chunk_type}):\n{chunk}")
    return "\n\n".join(numbered)

MAX_CONTEXT_TOKENS = 3000

def truncate_context_by_tokens(chunks_text: str, max_tokens: int = MAX_CONTEXT_TOKENS) -> str:
    """Truncates context if it exceeds maximum token limit."""
    token_ids = tokenizer.encode(chunks_text, add_special_tokens=False)
    if len(token_ids) <= max_tokens:
        return chunks_text
    truncated_ids = token_ids[:max_tokens]
    truncated_text = tokenizer.decode(truncated_ids, skip_special_tokens=True)
    return truncated_text + "\n\n[...context κόπηκε λόγω μεγέθους...]"

print("Οι συναρτήσεις φορτώθηκαν επιτυχώς.")


In [ ]:
# 7. Execute RAG evaluation loop with checkpointing to Google Drive
from tqdm import tqdm
import torch
import gc
import os
import json as _json

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

CHECKPOINT_DIR = "/content/drive/MyDrive/rag_eval_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "evaluation_checkpoint.json")

# Load previous checkpoint if available
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        results = _json.load(f)
    done_idx = {r["idx"] for r in results}
    print(f"Βρέθηκε checkpoint με {len(results)} ήδη αξιολογημένες ερωτήσεις. Συνέχιση εκτέλεσης.")
else:
    results = []
    done_idx = set()
    print("Δεν βρέθηκε προηγούμενο checkpoint. Έναρξη αξιολόγησης από την αρχή.")

SAVE_EVERY = 1

# Evaluation loop execution
for idx, item in enumerate(tqdm(data, desc="Αξιολόγηση")):
    current_idx = idx + 1

    if current_idx in done_idx:
        continue

    question = item["query"]
    answer   = item["generated_answer"]
    category = item["_category"]
    chunks_text = prepare_context_for_prompt(item["retrieved_context"])
    chunks_text = truncate_context_by_tokens(chunks_text)
    special  = SPECIAL_INSTRUCTIONS.get(category, "")

    entry = {
        "idx":         current_idx,
        "query":       question,
        "category":    category,
        "num_chunks":  len(item["_chunks"]),
        "has_images":  item["_has_image_chunks"],
        "answer":      answer,
    }

    oom_occurred = False

    # Evaluate Faithfulness
    try:
        faith_raw = run_judge(FAITHFULNESS_PROMPT.format(
            question=question,
            chunks=chunks_text,
            answer=answer,
            special_instructions=special
        ))
        faith_analysis, faith_score = parse_score_and_analysis(faith_raw)
    except torch.cuda.OutOfMemoryError:
        print(f"Σφάλμα OOM (Faithfulness) στην ερώτηση #{current_idx}: {question[:60]}... (παραλείπεται)")
        torch.cuda.empty_cache(); gc.collect()
        faith_raw, faith_analysis, faith_score = "(OOM)", "(OOM - παραλείφθηκε)", -1
        oom_occurred = True
    entry["faithfulness_score"]    = faith_score
    entry["faithfulness_analysis"] = faith_analysis
    entry["faithfulness_raw"]      = faith_raw

    # Evaluate Answer Relevance
    try:
        rel_raw = run_judge(RELEVANCE_PROMPT.format(
            question=question,
            answer=answer,
            special_instructions=special
        ))
        rel_analysis, rel_score = parse_score_and_analysis(rel_raw)
    except torch.cuda.OutOfMemoryError:
        print(f"Σφάλμα OOM (Relevance) στην ερώτηση #{current_idx}: {question[:60]}... (παραλείπεται)")
        torch.cuda.empty_cache(); gc.collect()
        rel_raw, rel_analysis, rel_score = "(OOM)", "(OOM - παραλείφθηκε)", -1
        oom_occurred = True
    entry["relevance_score"]    = rel_score
    entry["relevance_analysis"] = rel_analysis
    entry["relevance_raw"]      = rel_raw

    # Evaluate Context Precision/Recall
    try:
        ctx_raw = run_judge(CONTEXT_PROMPT.format(
            question=question,
            chunks=chunks_text,
            special_instructions=special
        ))
        ctx_analysis, ctx_score = parse_score_and_analysis(ctx_raw)
    except torch.cuda.OutOfMemoryError:
        print(f"Σφάλμα OOM (Context) στην ερώτηση #{current_idx}: {question[:60]}... (παραλείπεται)")
        torch.cuda.empty_cache(); gc.collect()
        ctx_raw, ctx_analysis, ctx_score = "(OOM)", "(OOM - παραλείφθηκε)", -1
        oom_occurred = True
    entry["context_score"]    = ctx_score
    entry["context_analysis"] = ctx_analysis
    entry["context_raw"]      = ctx_raw

    # Calculate Composite Score
    valid = [s for s in [faith_score, rel_score, ctx_score] if s != -1]
    entry["composite_score"] = round(sum(valid) / len(valid), 2) if valid else -1

    results.append(entry)
    done_idx.add(current_idx)

    print(f"[{current_idx}/{len(data)}] {question[:60]}...")
    print(f"  Category: {category} | Chunks: {len(item['_chunks'])} | "
          f"F:{faith_score} R:{rel_score} C:{ctx_score} -> {entry['composite_score']}"
          + (" (Σφάλμα OOM σε κάποιο metric)" if oom_occurred else ""))

    torch.cuda.empty_cache()
    gc.collect()

    if len(results) % SAVE_EVERY == 0:
        with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
            _json.dump(results, f, ensure_ascii=False, indent=2)

with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
    _json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nΗ αξιολόγηση ολοκληρώθηκε επιτυχώς! ({len(results)}/{len(data)} ερωτήσεις)")
print(f"Checkpoint αποθηκευμένο στο: {CHECKPOINT_PATH}")


In [ ]:
# 8. Calculate and display overall and per-category evaluation metrics
df = pd.DataFrame(results)

print("=" * 60)
print("ΣΥΝΟΛΙΚΑ ΑΠΟΤΕΛΕΣΜΑΤΑ")
print("=" * 60)
metrics = ["faithfulness_score", "relevance_score", "context_score", "composite_score"]
labels  = ["Faithfulness", "Answer Relevance", "Context P/R", "Composite"]
for metric, label in zip(metrics, labels):
    valid = df[df[metric] != -1][metric]
    if len(valid) > 0:
        print(f"  {label:20s}: {valid.mean():.2f}/5.00  (min={valid.min()}, max={valid.max()})")

print()
print("=" * 60)
print("ΑΝΑ ΚΑΤΗΓΟΡΙΑ")
print("=" * 60)
for cat in df["category"].unique():
    cat_df = df[df["category"] == cat]
    print(f"\nΚατηγορία: {cat.upper()} ({len(cat_df)} ερωτήσεις):")
    for metric, label in zip(metrics[:3], labels[:3]):
        valid = cat_df[cat_df[metric] != -1][metric]
        if len(valid) > 0:
            print(f"   {label:20s}: {valid.mean():.2f}")

print()
print("=" * 60)
print("ΠΙΟ ΑΔΥΝΑΜΕΣ ΑΠΑΝΤΗΣΕΙΣ (composite < 3)")
print("=" * 60)
weak = df[df["composite_score"] < 3].sort_values("composite_score")
if len(weak) == 0:
    print("  Καμία απάντηση κάτω από 3/5!")
for _, row in weak.iterrows():
    print(f"  [{row['idx']}] ({row['category']}) score={row['composite_score']}: {row['query'][:65]}...")


In [ ]:
# 9. Detailed evaluation inspection for a single response
INSPECT_IDX = 1  # Select question index (1-indexed)

row = df[df["idx"] == INSPECT_IDX].iloc[0]

print("=" * 70)
print(f"ΕΡΩΤΗΣΗ #{INSPECT_IDX} - Κατηγορία: {row['category'].upper()}")
print("=" * 70)
print(f"\nΕΡΩΤΗΣΗ:\n{row['query']}")
print(f"\nΑΠΑΝΤΗΣΗ ΣΥΣΤΗΜΑΤΟΣ:\n{row['answer'][:600]}{'...' if len(row['answer']) > 600 else ''}")
print("\n" + "-" * 70)
print(f"FAITHFULNESS ({row['faithfulness_score']}/5)")
print(f"   {row['faithfulness_analysis']}")
print(f"\nANSWER RELEVANCE ({row['relevance_score']}/5)")
print(f"   {row['relevance_analysis']}")
print(f"\nCONTEXT PRECISION/RECALL ({row['context_score']}/5)")
print(f"   {row['context_analysis']}")
print(f"\nCOMPOSITE: {row['composite_score']}/5")


In [ ]:
# 10. Export detailed evaluation results to JSON and summary CSV files
from google.colab import files

output_data = []
for r in results:
    output_data.append({
        "idx":                  r["idx"],
        "query":                r["query"],
        "category":             r["category"],
        "num_chunks":           r["num_chunks"],
        "has_image_chunks":     r["has_images"],
        "faithfulness_score":   r["faithfulness_score"],
        "faithfulness_analysis": r["faithfulness_analysis"],
        "relevance_score":      r["relevance_score"],
        "relevance_analysis":   r["relevance_analysis"],
        "context_score":        r["context_score"],
        "context_analysis":     r["context_analysis"],
        "composite_score":      r["composite_score"],
        "faithfulness_raw":     r["faithfulness_raw"],
        "relevance_raw":        r["relevance_raw"],
        "context_raw":          r["context_raw"],
    })

output_json = "/content/evaluation_results.json"
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)
print(f"JSON: {output_json}")

output_csv = "/content/evaluation_summary.csv"
summary_cols = [
    "idx", "category", "query",
    "faithfulness_score", "faithfulness_analysis",
    "relevance_score",    "relevance_analysis",
    "context_score",      "context_analysis",
    "composite_score",    "num_chunks", "has_images"
]
df[summary_cols].to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"CSV: {output_csv}")

files.download(output_json)
files.download(output_csv)


In [ ]:
# 11. Generate visualization charts for evaluation metrics
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Average per criterion
metric_means = [
    df[df["faithfulness_score"] != -1]["faithfulness_score"].mean(),
    df[df["relevance_score"]    != -1]["relevance_score"].mean(),
    df[df["context_score"]      != -1]["context_score"].mean(),
]
bars = axes[0].bar(
    ["Faithfulness", "Answer\nRelevance", "Context\nP/R"],
    metric_means,
    color=["#4C72B0", "#DD8452", "#55A868"],
    edgecolor="white", width=0.5
)
axes[0].set_ylim(0, 5)
axes[0].set_ylabel("Μέσος όρος (1-5)")
axes[0].set_title("Συνολική Αξιολόγηση ανά Κριτήριο")
axes[0].axhline(y=3, color='red', linestyle='--', alpha=0.5, label='Όριο 3/5')
for bar, val in zip(bars, metric_means):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
                f'{val:.2f}', ha='center', va='bottom', fontweight='bold')
axes[0].legend()

# Chart 2: Composite Score per category
cat_means = df.groupby("category")["composite_score"].mean().sort_values()
axes[1].barh(cat_means.index, cat_means.values, color="#8172B2", edgecolor="white")
axes[1].set_xlim(0, 5)
axes[1].set_xlabel("Composite Score (1-5)")
axes[1].set_title("Composite Score ανά Κατηγορία")
axes[1].axvline(x=3, color='red', linestyle='--', alpha=0.5)
for i, (cat, val) in enumerate(cat_means.items()):
    axes[1].text(val + 0.05, i, f'{val:.2f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig("/content/evaluation_chart.png", dpi=150, bbox_inches='tight')
plt.show()

files.download("/content/evaluation_chart.png")
